In [ ]:
# Basic imports
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

# Visuallization style 
plt.style.use('seaborn-v0_8')

# Check current working directory
print(os.getcwd())
print(os.listdir('..'))

In [ ]:
# Load dataset
df = pd.read_csv('C:\\Users\\VRAJPBH\\OneDrive - Pearson\\Documents\\GitHub\\Loan-Default-Risk\\data\\credit_risk_dataset.csv')
df.head()

In [ ]:
# Dataset overview
print("Shape:", df.shape)
df.info()

In [ ]:
# Display column names
df.columns

# Define target variable (loan_status: 1 - Default, 0 - No Default)
df['loan_status'].value_counts(normalize=True) 

In [ ]:
# Target distribution visualization
sns.countplot(x='loan_status', data=df)
plt.title('Loan Default Distribution')
plt.show()

### Target Variable Insight

The dataset shows class imbalance, with fewer loan defaults compared to non-defaults.
Therefore, evaluation metrics such as recall, F1-score, and ROC-AUC are more appropriate
than accuracy for assessing model performance.


In [ ]:
# Missing values analysis
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Fill numerical missing values with median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical missing values with mode
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [ ]:
# Encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

In [ ]:
# Split features and target variable
X = df_encoded.drop('loan_status', axis=1)
y = df_encoded['loan_status']


In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [ ]:
# Baseline model with Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_lr))

### Baseline Model Insight

Logistic Regression provides a strong baseline for loan default prediction.
However, due to non-linear relationships in financial data, more advanced models
may improve performance.


In [ ]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_clf.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_rf = rf_clf.predict(X_test)
y_proba_rf = rf_clf.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate random forest model
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(classification_report(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_rf))

# Confusion Matrix
confusion_matrix(y_test, y_pred_rf)

### Model Comparison

Random Forest outperformed Logistic Regression by capturing non-linear relationships
between borrower attributes and default risk. It achieved a higher ROC-AUC and
improved recall for default cases, making it more suitable for financial risk prediction.


In [ ]:
# Feature importance
feature_importances = pd.Series(
    rf_clf.feature_importances_,
    index=X.columns).sort_values(ascending=False)

feature_importances.head(10)


In [ ]:
# Plot top feature 
feature_importances.head(10).plot(
    kind='barh',
    title='Top 10 Feature Influencing Loan Default Risk'
)

plt.gca().invert_yaxis()
plt.show()

### Key Risk Drivers

The most influential factors in predicting loan default include loan interest rate,
loan amount, borrower income, employment length, and credit history indicators.
These features can help financial institutions assess borrower risk more effectively.
